# 12. A closed adaptive optics loop

The turbulence of the atmosphere corrugates the wavefront of a star; a deformable mirror, a wavefront sensor
and a controller correct it in a closed loop. LUMOS keeps the graph of a context acyclic: the loop is a series
of time iterations, where an output (the commands computed by the controller) is **fed back** to a field (the
commands of the mirror) at the next step (ADR-0020). The preset `adaptive_optics.lumos` observes a star in the
H band with an 8.2 m telescope through turbulence in frozen flow ($r_0$ = 15 cm at 500 nm, wind 10 m/s).

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import numpy as np
from astropy import units as u

import lumos
from lumos import plotting
from lumos.plotting import style

style.apply()
ctx = lumos.io.load("presets/adaptive_optics.lumos")
print(ctx.feedbacks)

## Open loop

Before the loop runs, the mirror is flat: the image of the star is a speckle cloud spread over
$\lambda / r_0$ instead of $\lambda / D$.

In [ ]:
open_loop = ctx.run(["camera.out", "wfs.residual"])
print("residual", open_loop["wfs.residual"].data)
fig, axes = style.figure("double", ncols=2, aspect=0.45)
plotting.plot_data(open_loop["camera.out"], ax=axes[0])
axes[0].set_title("open loop")

## Closing the loop

The controller is an integrator: $\Delta c = -g R s$, with $s$ the slopes measured by the Shack-Hartmann
sensor and $R$ the regularised pseudo-inverse of the interaction matrix of the mirror. Each iteration of
2 ms computes the outputs, then adds $\Delta c$ to the commands of the mirror, then advances the clock: the
correction acts one frame later. With a gain $g$ the error of a static aberration decreases as $(1 - g)^k$;
on turbulence the residual settles near the fitting error of the mirror, about $0.3 (d / r_0)^{5/3}$ rad² for
an actuator pitch $d$ (Hudgin 1977), plus the error due to the wind during the delay.

In [ ]:
steps = ctx.loop(40, period=2 * u.ms, outputs=["wfs.residual", "camera.out"])
time = np.array([s.time.to_value(u.ms) for s in steps])
residual = np.array([s.outputs["wfs.residual"].data.to_value(u.nm) for s in steps])

wavelength = 1.65e-6
r0 = 0.15 * (wavelength / 0.5e-6) ** 1.2
pitch = 8.2 / 16
fitting = np.sqrt(0.3 * (pitch / r0) ** (5 / 3)) * wavelength / (2 * np.pi) * 1e9
strehl = np.exp(-((2 * np.pi * residual * 1e-9 / wavelength) ** 2))
print(f"residual {residual[0]:.0f} nm -> {residual[-1]:.0f} nm (fitting error about {fitting:.0f} nm)")
print(f"Strehl ratio (Marechal) {strehl[-1]:.2f}")

fig2, ax = style.figure("single")
ax.plot(time, residual, "o-")
ax.axhline(fitting, color="C1", ls="--", label="fitting error")
ax.set_xlabel("time [ms]")
ax.set_ylabel("residual [nm]")
ax.legend();

## Closed loop

The image of the star is now close to the diffraction limit of the telescope; the loop has kept its state:
the mirror holds the last commands.

In [ ]:
closed = ctx.run(ctx.meta["outputs"], archive="adaptive-optics")
plotting.plot_data(closed["camera.out"], ax=axes[1])
axes[1].set_title("closed loop")
ratio = closed["camera.out"].data.value.max() / open_loop["camera.out"].data.value.max()
print(f"peak of the image x{ratio:.1f}")
closed.archive.save_figure(fig, "adaptive_optics_psf")
closed.archive.save_figure(fig2, "adaptive_optics_residual")
fig